# Cache Mask2Former Models (Notebook Workflow)

This notebook mirrors the `download_models.py` script and caches Mask2Former models from Hugging Face into the local transformers cache.

## 1) Set Up Notebook Environment and Imports

In [ ]:
import sys
import subprocess
from pathlib import Path
from typing import Iterable, List

# Optional lightweight dependency check for notebook use.
def _pip_install_if_missing(import_name: str, pip_name: str | None = None) -> None:
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except Exception:
        print(f"[install] {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

_pip_install_if_missing("torch")
_pip_install_if_missing("transformers")

from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

ROOT = Path("/home/abdou/Projects/Python/RTSGS").resolve()
print("Workspace root:", ROOT)
print("Python:", sys.executable)

## 2) Define Default Model IDs and User-Provided Model List

In [ ]:
DEFAULT_MODELS = [
    "facebook/mask2former-swin-base-coco-panoptic",
    "facebook/mask2former-swin-base-ade-semantic",
]

# Simulates repeated --model CLI arguments.
CUSTOM_MODELS = [
    # "facebook/mask2former-swin-small-ade-semantic",
    "facebook/mask2former-swin-base-ade-semantic",
    "  ",
]

INCLUDE_ADE = True

print("Default:", DEFAULT_MODELS)
print("Custom:", CUSTOM_MODELS)
print("Include ADE flag:", INCLUDE_ADE)

## 3) Build Stable De-duplication Logic for Model IDs

In [ ]:
def dedupe_model_ids(model_ids: Iterable[str]) -> List[str]:
    deduped: List[str] = []
    seen: set[str] = set()
    for model_id in model_ids:
        model_id = str(model_id).strip()
        if not model_id or model_id in seen:
            continue
        seen.add(model_id)
        deduped.append(model_id)
    return deduped

all_ids = list(DEFAULT_MODELS)
all_ids.extend(CUSTOM_MODELS)
if INCLUDE_ADE:
    all_ids.append("facebook/mask2former-swin-base-ade-semantic")

DEDUPED_MODEL_IDS = dedupe_model_ids(all_ids)
print("Final model list:")
for m in DEDUPED_MODEL_IDS:
    print(" -", m)

## 4) Implement `cache_model()` for Processor and Weights

In [ ]:
def cache_model(model_id: str) -> bool:
    print(f"[cache] Downloading processor for {model_id} ...")
    AutoImageProcessor.from_pretrained(model_id)

    print(f"[cache] Downloading model weights for {model_id} ...")
    Mask2FormerForUniversalSegmentation.from_pretrained(model_id)

    print(f"[cache] Ready: {model_id}")
    return True

## 5) Run Batch Caching with Failure Tracking

In [ ]:
if not DEDUPED_MODEL_IDS:
    raise RuntimeError("No model ids provided.")

failures = 0
for model_id in DEDUPED_MODEL_IDS:
    try:
        cache_model(model_id)
    except Exception as exc:
        failures += 1
        print(f"[cache] Failed: {model_id} -> {exc}")

if failures:
    print(f"Finished with {failures} failure(s).")
else:
    print("All models cached successfully.")

## 6) Validate Cached Artifacts by Reloading a Model

In [ ]:
verify_model_id = "facebook/mask2former-swin-base-ade-semantic"
processor = AutoImageProcessor.from_pretrained(verify_model_id, local_files_only=True)
model = Mask2FormerForUniversalSegmentation.from_pretrained(verify_model_id, local_files_only=True)

print("Verified cached model:", verify_model_id)
print("Model type:", model.config.model_type)
print("Num labels:", model.config.num_labels)
print("ID2LABEL sample:", list(model.config.id2label.items())[:8])
print("Processor loaded:", processor.__class__.__name__)